# 02 · Generación y carga de datos sintéticos

Genera con **Faker** un volumen de datos realista para VoltiaTech y lo carga en las tablas
creadas en el notebook anterior.

**Volumen objetivo del enunciado:**

| Tabla | Mínimo pedido |
|---|---|
| `customers` | 500 |
| `products` | 70 |
| `orders` | 2.000 |
| `order_items` | ~4.500 (2-3 por pedido) |
| `payments` | uno por pedido |
| `reviews` | ~35% de las líneas entregadas |

**El requisito difícil no es el volumen, es la coherencia de negocio:** que las fechas vayan
en orden, que el importe del pago cuadre con las líneas, que no se valore un pedido que
nunca llegó. Eso es lo que se cuida aquí.

## 1. Imports y conexión

In [1]:
import os
import sys

# el proyecto se importa desde la raiz del repo, no desde notebooks/
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))   # si no existe, no pasa nada
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from src.esquema import ESQUEMA, ORDEN_CARGA, campos, ddl_sqlite, foraneas, resumen
from src.conexion import obtener_backend

import random
from datetime import date, timedelta

import numpy as np
from faker import Faker

SEMILLA = 42
Faker.seed(SEMILLA)
random.seed(SEMILLA)
np.random.seed(SEMILLA)

fake = Faker("es_ES")
backend = obtener_backend()

Backend: SQLite (espejo local). Motivo: GCP_PROJECT_ID no está configurado en el .env.
Las queries son SQL estándar: el resultado es equivalente al de BigQuery.


## 2. Parámetros del negocio

Los saco a constantes para que las decisiones de negocio queden explícitas y se puedan
cambiar en un solo sitio.

In [2]:
N_CUSTOMERS = 500
N_PRODUCTS = 70
N_ORDERS = 2000

PAISES = ["España", "Francia", "Alemania", "Italia", "Portugal", "Países Bajos", "Bélgica"]
CANALES = ["organic", "paid_ads", "social", "email", "referral", "marketplace"]

# estado del pedido y su peso: la mayoria de pedidos acaban entregados
ESTADOS_PEDIDO = {
    "delivered": 0.62, "shipped": 0.12, "confirmed": 0.08,
    "pending": 0.06, "cancelled": 0.07, "returned": 0.05,
}
METODOS_PAGO = ["card", "paypal", "bank_transfer", "bizum"]

# rango de precio y margen tipico por categoria, para que los precios tengan sentido
CATEGORIAS = {
    "Smartphones":  ("Teléfonos móviles y accesorios", 180, 1400, 0.18),
    "Laptops":      ("Portátiles y ultrabooks",        400, 2600, 0.15),
    "Audio":        ("Auriculares, altavoces y sonido", 25,  600, 0.35),
    "Wearables":    ("Relojes y pulseras inteligentes", 40,  800, 0.30),
    "Periféricos":  ("Teclados, ratones y monitores",   15,  900, 0.32),
    "Tablets":      ("Tablets y lápices digitales",    120, 1500, 0.20),
    "Gaming":       ("Consolas y accesorios de juego",  30, 1200, 0.22),
    "Smart Home":   ("Domótica y asistentes de voz",    20,  500, 0.38),
}

FECHA_INICIO = date(2024, 1, 1)
FECHA_FIN = date(2025, 12, 31)

print(f"{len(CATEGORIAS)} categorías | {N_PRODUCTS} productos | "
      f"{N_CUSTOMERS} clientes | {N_ORDERS} pedidos")

8 categorías | 70 productos | 500 clientes | 2000 pedidos


## 3. `categories` y `products`

El **coste** se calcula a partir del precio y del margen típico de la categoría, con algo de
ruido. Así `price > cost` siempre (un producto no se vende a pérdida) y el margen resultante
es distinto por categoría, que es lo que hace interesante el análisis posterior.

In [3]:
# --- categories ---
categories = pd.DataFrame([
    {"category_id": i, "category_name": nombre, "description": desc}
    for i, (nombre, (desc, _, _, _)) in enumerate(CATEGORIAS.items(), start=1)
])

# --- products ---
MODELOS = ["Pro", "Air", "Max", "Lite", "Ultra", "Plus", "Mini", "Edge", "Neo", "One"]
MARCAS = ["Voltia", "Nexor", "Krono", "Aurio", "Zentra", "Lumix", "Orbit"]

filas = []
for pid in range(1, N_PRODUCTS + 1):
    nombre_cat = random.choice(list(CATEGORIAS))
    _, minimo, maximo, margen = CATEGORIAS[nombre_cat]
    cat_id = int(categories.loc[categories["category_name"] == nombre_cat, "category_id"].iloc[0])

    precio = round(random.uniform(minimo, maximo), 2)
    # el coste sale del margen de la categoria, con +-4 puntos de ruido
    margen_real = min(max(margen + random.uniform(-0.04, 0.04), 0.05), 0.60)
    coste = round(precio * (1 - margen_real), 2)

    filas.append({
        "product_id": pid,
        "product_name": f"{random.choice(MARCAS)} {random.choice(MODELOS)} {random.randint(1, 15)}",
        "category_id": cat_id,
        "price": precio,
        "cost": coste,
        "stock": random.randint(0, 400),
        # un 8% del catalogo esta descatalogado
        "is_active": random.random() > 0.08,
    })

products = pd.DataFrame(filas)

print(f"categories: {len(categories)} | products: {len(products)}")
assert (products["price"] > products["cost"]).all(), "Ningún producto puede venderse por debajo de coste"
display(products.head(3))

categories: 8 | products: 70


,product_id,product_name,category_id,price,cost,stock,is_active
0,1,Nexor Max 12,2,455.02,394.96,52,True
1,2,Voltia Lite 4,2,1699.08,1507.86,258,True
2,3,Aurio Lite 8,4,584.17,399.51,301,True


## 4. `customers`

Los países no se reparten de forma uniforme: España pesa más porque es el mercado de origen.
Un reparto uniforme daría un análisis geográfico plano y poco realista.

In [4]:
PESOS_PAIS = [0.42, 0.14, 0.14, 0.11, 0.08, 0.06, 0.05]

filas = []
for cid in range(1, N_CUSTOMERS + 1):
    nombre = fake.first_name()
    apellido = fake.last_name()
    filas.append({
        "customer_id": cid,
        "first_name": nombre,
        "last_name": apellido,
        # email derivado del nombre + el id, para que no haya duplicados
        "email": f"{nombre.lower()}.{apellido.lower().replace(' ', '')}{cid}@example.com",
        "country": random.choices(PAISES, weights=PESOS_PAIS)[0],
        "city": fake.city(),
        "acquisition_channel": random.choices(
            CANALES, weights=[0.30, 0.22, 0.18, 0.12, 0.10, 0.08])[0],
        "registered_at": fake.date_between(FECHA_INICIO, FECHA_FIN).isoformat(),
    })

customers = pd.DataFrame(filas)

print(f"customers: {len(customers)}")
assert customers["email"].is_unique, "Los emails deben ser únicos"
display(customers["country"].value_counts().to_frame("clientes"))

customers: 500


,clientes
country,
España,197
Alemania,78
Francia,70
Italia,60
Portugal,48
Países Bajos,24
Bélgica,23


## 5. `orders`

Aquí está la coherencia temporal, que es lo que separa un dataset creíble de uno inventado:

- Un cliente **no puede pedir antes de registrarse**: `ordered_at >= registered_at`.
- `shipped_at` solo existe si el pedido salió (`shipped`, `delivered` o `returned`).
- `delivered_at` solo existe si llegó, y siempre **después** de `shipped_at`.
- Un pedido `pending` o `cancelled` no tiene ninguna de las dos fechas.

In [5]:
estados = list(ESTADOS_PEDIDO)
pesos_estado = list(ESTADOS_PEDIDO.values())

filas = []
for oid in range(1, N_ORDERS + 1):
    cliente = customers.sample(1).iloc[0]
    alta = date.fromisoformat(cliente["registered_at"])

    # el pedido nunca puede ser anterior al alta del cliente
    if alta >= FECHA_FIN:
        pedido = FECHA_FIN
    else:
        pedido = fake.date_between(alta, FECHA_FIN)

    estado = random.choices(estados, weights=pesos_estado)[0]

    enviado = entregado = None
    if estado in ("shipped", "delivered", "returned"):
        enviado = pedido + timedelta(days=random.randint(1, 4))
    if estado in ("delivered", "returned"):
        entregado = enviado + timedelta(days=random.randint(1, 7))

    filas.append({
        "order_id": oid,
        "customer_id": int(cliente["customer_id"]),
        "status": estado,
        "shipping_country": cliente["country"],
        "shipping_city": cliente["city"],
        "ordered_at": pedido.isoformat(),
        "shipped_at": enviado.isoformat() if enviado else None,
        "delivered_at": entregado.isoformat() if entregado else None,
    })

orders = pd.DataFrame(filas)

print(f"orders: {len(orders)}")
display(orders["status"].value_counts().to_frame("pedidos"))

orders: 2000


,pedidos
status,
delivered,1268
shipped,247
confirmed,138
cancelled,132
returned,109
pending,106


## 6. `order_items`

**El punto más importante del modelo.** `unit_price` guarda el precio **del momento de la
compra**, no se lee de `products.price`. Si mañana sube el precio del catálogo, los pedidos
históricos deben seguir valiendo lo que valieron: si no, el análisis de ingresos pasados se
reescribiría solo. Por eso simulo una pequeña variación respecto al precio actual (promociones,
cambios de tarifa).

In [6]:
filas = []
item_id = 1
for oid in orders["order_id"]:
    # entre 1 y 5 lineas por pedido, con 2-3 como caso mas frecuente
    n_lineas = random.choices([1, 2, 3, 4, 5], weights=[0.20, 0.33, 0.27, 0.13, 0.07])[0]
    productos_pedido = products.sample(n_lineas)  # sin repetir producto en el mismo pedido

    for _, prod in productos_pedido.iterrows():
        # el precio pagado se parece al actual, pero no es el actual
        precio_pagado = round(float(prod["price"]) * random.uniform(0.92, 1.05), 2)
        # el 78% de las lineas no lleva descuento
        descuento = 0.0 if random.random() < 0.78 else float(random.choice([5, 10, 15, 20, 25]))

        filas.append({
            "order_item_id": item_id,
            "order_id": int(oid),
            "product_id": int(prod["product_id"]),
            "quantity": random.choices([1, 2, 3], weights=[0.75, 0.19, 0.06])[0],
            "unit_price": precio_pagado,
            "discount_pct": descuento,
        })
        item_id += 1

order_items = pd.DataFrame(filas)

print(f"order_items: {len(order_items)} | media por pedido: "
      f"{len(order_items) / len(orders):.2f}")
display(order_items.head(3))

order_items: 5113 | media por pedido: 2.56


,order_item_id,order_id,product_id,quantity,unit_price,discount_pct
0,1,1,47,1,261.91,0.0
1,2,1,31,1,1346.43,15.0
2,3,2,11,3,473.42,0.0


## 7. `payments`

El importe del pago **se calcula sumando las líneas del pedido** con su descuento aplicado.
No es un número al azar: si no cuadrara con `order_items`, cualquier query de ingresos daría
resultados contradictorios según por dónde se calcule.

El estado del pago depende del estado del pedido: un pedido devuelto tiene un pago
reembolsado, uno pendiente no está cobrado todavía.

In [7]:
# importe real de cada pedido, a partir de sus lineas
importes = (
    order_items
    .assign(total_linea=lambda d: d["unit_price"] * d["quantity"] * (1 - d["discount_pct"] / 100))
    .groupby("order_id")["total_linea"].sum()
    .round(2)
)

filas = []
for i, pedido in enumerate(orders.itertuples(), start=1):
    importe = float(importes.get(pedido.order_id, 0.0))

    # el estado del pago se deriva del estado del pedido
    if pedido.status == "returned":
        estado_pago, cobrado = "refunded", True
    elif pedido.status == "cancelled":
        estado_pago, cobrado = random.choice([("failed", False), ("refunded", True)])[0], False
        estado_pago = random.choice(["failed", "refunded"])
        cobrado = estado_pago == "refunded"
    elif pedido.status == "pending":
        estado_pago, cobrado = "pending", False
    else:
        estado_pago, cobrado = "completed", True

    filas.append({
        "payment_id": i,
        "order_id": pedido.order_id,
        "payment_method": random.choices(METODOS_PAGO, weights=[0.55, 0.22, 0.13, 0.10])[0],
        "payment_status": estado_pago,
        "amount": round(importe, 2),
        # se cobra el mismo dia del pedido o al siguiente
        "paid_at": (date.fromisoformat(pedido.ordered_at)
                    + timedelta(days=random.randint(0, 1))).isoformat() if cobrado else None,
    })

payments = pd.DataFrame(filas)

print(f"payments: {len(payments)}")
display(payments["payment_status"].value_counts().to_frame("pagos"))

payments: 2000


,pagos
payment_status,
completed,1653
refunded,176
pending,106
failed,65


## 8. `reviews`

**Solo se puede valorar lo que se ha recibido.** Filtro las líneas de pedidos `delivered` y
tomo de ahí un 35%. Valorar un pedido cancelado sería un dato imposible que rompería
cualquier análisis de satisfacción.

Las notas no son uniformes: la mayoría de la gente que se molesta en valorar pone 4 o 5.

In [8]:
COMENTARIOS = {
    5: ["Perfecto, justo lo que esperaba.", "Muy buena calidad y llegó antes de tiempo.",
        "Repetiré sin duda."],
    4: ["Buen producto, aunque el embalaje llegó algo dañado.", "Cumple lo que promete.",
        "Contento con la compra."],
    3: ["Correcto por el precio, pero nada del otro mundo.", "Esperaba algo más.",
        "Cumple, sin más."],
    2: ["La batería dura bastante menos de lo anunciado.", "Se calienta demasiado.",
        "No lo recomendaría."],
    1: ["Llegó defectuoso.", "Nada que ver con la descripción.", "Muy decepcionado."],
}

entregados = orders.loc[orders["status"] == "delivered", "order_id"]
lineas_entregadas = order_items[order_items["order_id"].isin(entregados)]
valoradas = lineas_entregadas.sample(frac=0.35, random_state=SEMILLA)

fechas_entrega = orders.set_index("order_id")["delivered_at"].to_dict()

filas = []
for i, linea in enumerate(valoradas.itertuples(), start=1):
    nota = random.choices([1, 2, 3, 4, 5], weights=[0.04, 0.07, 0.14, 0.32, 0.43])[0]
    entrega = date.fromisoformat(fechas_entrega[linea.order_id])
    filas.append({
        "review_id": i,
        "order_item_id": linea.order_item_id,
        "rating": nota,
        # un 25% valora sin escribir comentario
        "comment": random.choice(COMENTARIOS[nota]) if random.random() > 0.25 else None,
        # se valora entre 1 y 30 dias despues de recibirlo
        "created_at": (entrega + timedelta(days=random.randint(1, 30))).isoformat(),
    })

reviews = pd.DataFrame(filas)

print(f"reviews: {len(reviews)} ({len(reviews) / len(lineas_entregadas) * 100:.1f}% "
      f"de las {len(lineas_entregadas)} líneas entregadas)")
display(reviews["rating"].value_counts().sort_index().to_frame("valoraciones"))

reviews: 1149 (35.0% de las 3283 líneas entregadas)


,valoraciones
rating,
1,50
2,68
3,146
4,374
5,511


## 9. Validación antes de cargar

Antes de tocar la base de datos compruebo la integridad del conjunto. Un `assert` que falla
aquí cuesta segundos; un dato corrupto ya cargado cuesta mucho más.

In [9]:
tablas = {
    "categories": categories, "customers": customers, "products": products,
    "orders": orders, "order_items": order_items, "payments": payments, "reviews": reviews,
}

# --- 1. integridad referencial: toda FK apunta a una PK existente ---
comprobaciones = [
    ("products.category_id -> categories", products["category_id"], categories["category_id"]),
    ("orders.customer_id -> customers",    orders["customer_id"],   customers["customer_id"]),
    ("order_items.order_id -> orders",     order_items["order_id"], orders["order_id"]),
    ("order_items.product_id -> products", order_items["product_id"], products["product_id"]),
    ("payments.order_id -> orders",        payments["order_id"],    orders["order_id"]),
    ("reviews.order_item_id -> order_items", reviews["order_item_id"], order_items["order_item_id"]),
]
print("--- Integridad referencial ---")
for nombre, origen, destino in comprobaciones:
    huerfanos = (~origen.isin(set(destino))).sum()
    print(f"  {nombre:<42} huérfanos: {huerfanos}")
    assert huerfanos == 0, f"Hay FKs huérfanas en {nombre}"

# --- 2. coherencia temporal ---
print("\n--- Coherencia temporal ---")
con_envio = orders.dropna(subset=["shipped_at", "delivered_at"])
assert (con_envio["delivered_at"] >= con_envio["shipped_at"]).all(), "Entrega anterior al envío"
assert (orders.dropna(subset=["shipped_at"])["shipped_at"]
        >= orders.dropna(subset=["shipped_at"])["ordered_at"]).all(), "Envío anterior al pedido"
fusion = orders.merge(customers[["customer_id", "registered_at"]], on="customer_id")
assert (fusion["ordered_at"] >= fusion["registered_at"]).all(), "Pedido anterior al alta del cliente"
print("  entrega >= envío >= pedido >= alta del cliente: OK")

# --- 3. coherencia de negocio ---
print("\n--- Coherencia de negocio ---")
assert (products["price"] > products["cost"]).all(), "Producto por debajo de coste"
assert reviews["rating"].between(1, 5).all(), "Rating fuera de 1-5"
assert order_items["discount_pct"].between(0, 100).all(), "Descuento fuera de 0-100"
pedidos_valorados = set(order_items.set_index("order_item_id")
                        .loc[reviews["order_item_id"], "order_id"])
estados_valorados = set(orders[orders["order_id"].isin(pedidos_valorados)]["status"])
assert estados_valorados == {"delivered"}, f"Se han valorado pedidos no entregados: {estados_valorados}"
print("  precio > coste, rating 1-5, descuentos 0-100, solo se valora lo entregado: OK")

print("\n--- Volumen final ---")
display(pd.DataFrame([{"tabla": n, "filas": len(d)} for n, d in tablas.items()]))

--- Integridad referencial ---
  products.category_id -> categories         huérfanos: 0
  orders.customer_id -> customers            huérfanos: 0
  order_items.order_id -> orders             huérfanos: 0
  order_items.product_id -> products         huérfanos: 0
  payments.order_id -> orders                huérfanos: 0
  reviews.order_item_id -> order_items       huérfanos: 0

--- Coherencia temporal ---
  entrega >= envío >= pedido >= alta del cliente: OK

--- Coherencia de negocio ---
  precio > coste, rating 1-5, descuentos 0-100, solo se valora lo entregado: OK

--- Volumen final ---


,tabla,filas
0,categories,8
1,customers,500
2,products,70
3,orders,2000
4,order_items,5113
5,payments,2000
6,reviews,1149


## 10. Carga

Se carga en el mismo `ORDEN_CARGA` que se crearon las tablas. Si se hiciera al revés, una
base de datos con las FK activadas rechazaría la inserción de `order_items` por apuntar a
pedidos que todavía no existen.

In [10]:
resultados = []
for tabla in ORDEN_CARGA:
    df = tablas[tabla][campos(tabla)]   # columnas en el orden del esquema
    cargadas = backend.cargar(tabla, df)
    resultados.append({"tabla": tabla, "filas_enviadas": len(df), "filas_en_destino": cargadas})
    print(f"  {tabla:<14} {len(df):>5} filas cargadas")

display(pd.DataFrame(resultados))

  categories         8 filas cargadas
  customers        500 filas cargadas
  products          70 filas cargadas
  orders          2000 filas cargadas
  order_items     5113 filas cargadas
  payments        2000 filas cargadas
  reviews         1149 filas cargadas


,tabla,filas_enviadas,filas_en_destino
0,categories,8,8
1,customers,500,500
2,products,70,70
3,orders,2000,2000
4,order_items,5113,5113
5,payments,2000,2000
6,reviews,1149,1149


## 11. Verificación de la carga

In [11]:
print("--- Recuento leído desde la base de datos ---")
for tabla in ORDEN_CARGA:
    n = backend.query(f"SELECT COUNT(*) AS n FROM {backend.tabla(tabla)}")["n"][0]
    esperado = len(tablas[tabla])
    estado = "OK" if n == esperado else "DESCUADRE"
    print(f"  {tabla:<14} {n:>5} filas (esperado {esperado})  {estado}")
    assert n == esperado

# prueba de fuego: un JOIN de las 4 tablas centrales tiene que devolver datos coherentes
prueba = backend.query(f"""
SELECT o.order_id,
       c.first_name || ' ' || c.last_name AS cliente,
       p.product_name,
       oi.quantity,
       oi.unit_price
FROM {backend.tabla('order_items')} oi
JOIN {backend.tabla('orders')}   o ON o.order_id = oi.order_id
JOIN {backend.tabla('customers')} c ON c.customer_id = o.customer_id
JOIN {backend.tabla('products')} p ON p.product_id = oi.product_id
LIMIT 5
""")
print("\n--- JOIN de comprobación sobre 4 tablas ---")
display(prueba)

--- Recuento leído desde la base de datos ---
  categories         8 filas (esperado 8)  OK
  customers        500 filas (esperado 500)  OK
  products          70 filas (esperado 70)  OK
  orders          2000 filas (esperado 2000)  OK
  order_items     5113 filas (esperado 5113)  OK
  payments        2000 filas (esperado 2000)  OK
  reviews         1149 filas (esperado 1149)  OK

--- JOIN de comprobación sobre 4 tablas ---


,order_id,cliente,product_name,quantity,unit_price
0,1,Marina Sanchez,Krono Pro 15,1,261.91
1,1,Marina Sanchez,Orbit Lite 3,1,1346.43
2,2,Ildefonso Peñalver,Krono Mini 5,3,473.42
3,3,Jose Francisco Coloma,Orbit Pro 2,1,449.01
4,3,Jose Francisco Coloma,Krono Mini 5,1,453.24


---

**Siguiente paso:** `03_queries_verification.ipynb` responde con SQL a las preguntas de negocio.

In [12]:
backend.cerrar()